# Reto 1 · Vuestras reglas, en vuestro proyecto (MIA)

## De qué va esto

En el ejercicio E2 del gimnasio lo teníais **todo en un solo notebook**: el modelo, las reglas y el motor.

En el Reto 1 vais a hacer **lo mismo**, pero con vuestro propio problema y **repartido en ficheros**. En las empresas se trabaja así por un motivo práctico: en el Reto 1 el mismo proyecto lo usan tres asignaturas, y cada una tiene que poder trabajar en su parte sin tocar las demás.

| Pieza | En el E2 estaba en… | Ahora está en… | ¿Quién la toca? |
|---|---|---|---|
| El modelo (lo que predice) | Paso 1: `probabilidad()` | `modelo.py` → `predecir(caso)` | SAA. Vosotros, cuando tengáis vuestro modelo |
| El manual de reglas | Paso 3: límites, `si_R1`… y la lista `reglas` | `mia/reglas.py` | **Vosotros (MIA)** |
| La recepcionista que aplica las reglas | Paso 4: la función `motor` | `mia/motor.py` | **Nadie**: ya está hecho |
| Probar socios | Paso 6: Nerea, Unai, Irati, Gorka | `mia/casos.py` | **Vosotros (MIA)** |
| La app y la API | — | `app/` y `api/` | Más adelante, en PIA |

**La buena noticia:** no hay nada nuevo que aprender. Es el mismo motor y son las mismas reglas del E2. Este notebook sirve para que veáis dónde está cada cosa y para que escribáis las vuestras.

**No necesitáis FastAPI.** Todo este notebook funciona sin él.

## Qué vais a hacer, en orden

1. Ver el camino de un caso, de principio a fin (pasos 1 y 2).
2. Probar varios casos de golpe (paso 3).
3. Cambiar un límite sin tocar el modelo (paso 4).
4. **Escribir vuestras propias reglas** (paso 5). Es lo importante.
5. Conectar vuestro modelo de SAA cuando lo tengáis (paso 6).

### Antes de empezar

- **Abrid este notebook desde la carpeta de la plantilla**, la que tiene `README.md`, `modelo.py` y la carpeta `mia`. Si no, Python no encuentra los ficheros.

- **Muy importante:** cuando cambiéis algo en un fichero `.py` (por ejemplo, `reglas.py`), el notebook **no se entera solo**. Hay que reiniciar el kernel (*Kernel → Restart*) y volver a ejecutar desde el Paso 1. Es el error más habitual.

> **Si te sale un error**
> - `ModuleNotFoundError: No module named 'modelo'` o `'mia'` → el notebook no está abierto desde la carpeta de la plantilla.
> - Cambias `reglas.py` y no cambia nada → reinicia el kernel.
> - `KeyError: 'algo'` → una regla lee un dato que el caso no tiene. Revisa el nombre (mayúsculas, tildes, guiones bajos).

## Paso 1. Traer las tres piezas

En el E2 escribíais todo en el notebook. Aquí cada pieza está en su fichero y la **importamos**: le decimos a Python *"trae la función `predecir` del fichero `modelo.py`"*. Es como `require` o `import` en JavaScript.

**Solo ejecútala.**

In [1]:
#Cuando ejecutas esta línea, Python ejecuta el fichero modelo.py entero, de arriba abajo, una vez. 
# Dentro de ese fichero hay esta línea, fuera de cualquier función: modelo = _cargar_o_entrenar()
#La variable modelo se queda en memoria mientras el notebook estaba abierto.

# modelo.py es el fichero.
# modelo, dentro de ese fichero, es la variable con el modelo entrenado.

from modelo import predecir              # SAA: el modelo


from mia.reglas import reglas             # MIA: vuestras reglas
from mia.motor import motor               # MIA: el motor
import pandas as pd

print('Hay', len(reglas), 'reglas en mia/reglas.py')

Hay 12 reglas en mia/reglas.py


Debería salir: `Hay 8 reglas en mia/reglas.py`. Son las 8 reglas del gimnasio del E2. Más adelante las cambiaréis por las vuestras.

## Paso 2. Un caso, de principio a fin

Esto es lo más importante del notebook. **Todo el Reto 1 se resume en estas tres líneas.** Son las mismas que usará la app de PIA:

1. **Los datos del caso**, en un diccionario (la ficha).
2. **El modelo predice**: `predecir()` calcula la probabilidad y la apuntamos en la ficha como `p` (**SAA**).
3. **Las reglas deciden**: `motor()` aplica las reglas y devuelve la decisión y el porqué (**MIA**).

`ver=True` hace que el motor cuente lo que hace en cada vuelta.

In [2]:
aitor = {'nombre': 'Aitor', 'visitas_mes': 2, 'meses_socio': 36}   # 1. los datos
aitor['p'] = predecir(aitor)   
print('Probabilidad de baja de Aitor:', aitor['p'])                    # 2. el modelo predice
resultado = motor(aitor, reglas, ver=True)                         # 3. las reglas deciden

Probabilidad de baja de Aitor: 0.8
Vuelta 1 : se cumplen ['R2'] -> se aplica R2 -> apunta {'riesgo': 'alto'}
Vuelta 2 : se cumplen ['R5'] -> se aplica R5 -> apunta {'decision': 'LLAMADA DE LA DIRECTORA'}

Aitor : LLAMADA DE LA DIRECTORA   (reglas usadas: R2 -> R5 )


### Cómo leer lo que ha salido

Es lo mismo que en el E2:

| Vuelta | ¿Qué se cumple? | ¿Qué se aplica? | ¿Qué apunta en la ficha? |
|---|---|---|---|
| 1 | R2 (p = 0,8 ≥ 0,7) | R2 | riesgo: alto |
| 2 | R5 (riesgo alto y 36 meses) | R5 | decisión: LLAMADA DE LA DIRECTORA |

R2 no decide nada: deja apuntado el riesgo. En la vuelta 2, R5 lee ese riesgo y decide. Eso es **encadenar reglas**.

### Qué hay dentro de `resultado`

El motor devuelve un **diccionario** con cuatro cosas. La celda de abajo las enseña.

In [3]:
print('Decisión:     ', resultado['decision'])
print('Reglas usadas:', resultado['usadas'])
print('¿Por qué?')
for linea in resultado['porque']:
    print('   ', linea)

Decisión:      LLAMADA DE LA DIRECTORA
Reglas usadas: ['R2', 'R5']
¿Por qué?
    R2: la probabilidad de baja es alta
    R5: riesgo alto y lleva 24 meses o más: es un socio fiel


Las frases del porqué salen del **quinto dato** de cada regla en `mia/reglas.py`. Es lo único nuevo respecto al E2:

`('R5', si_R5, {'decision': 'LLAMADA DE LA DIRECTORA'}, 1, 'riesgo alto y lleva 24 meses o más: es un socio fiel')`

**Para qué sirve:** cuando la directora (o un cliente) pregunte *"¿por qué le habéis llamado?"*, el sistema puede contestar con palabras. Es la pregunta 4 del RA5: *¿la app dice qué decisión toma y por qué?* Y es lo que pide la ley (RGPD) cuando una decisión automática afecta a una persona.

## Paso 3. Probar varios casos de golpe

En el E2 pasabais a los socios uno a uno y mirabais si salía lo que esperabais. Aquí lo hace el ordenador.

En `mia/casos.py` hay una lista de socios con la decisión que **esperamos**. La función `probar_casos()` pasa cada uno por el modelo y por el motor, y escribe **BIEN** si sale lo esperado o **MAL** si no.


In [4]:
#Trae la función `probar_casos` del fichero `mia/casos.py`. 
# En ese fichero está la lista `CASOS`: unos socios con la decisión que  esperamos que salga para cada uno.
from mia.casos import probar_casos


# `probar_casos(reglas, predecir, motor)` coge cada socio de la lista y hace lo mismo que hicimos con Aitor 
# en el Paso 2:
#    - le pide al modelo su probabilidad (`predecir`),
#    - lo pasa por el motor con nuestras reglas (`motor` y `reglas`),
#    - y compara la decisión que sale con la que esperábamos.



# Pasamos a probar_casos las reglas, la función del modelo y el motor
# reglas	Una lista de reglas (la que habéis escrito en mia/reglas.py)
# predecir	La función que devuelve la probabilidad de baja de un socio (la que habéis escrito en modelo.py)
# motor	    La función que aplica las reglas a cada socio

# (las funciones van sin paréntesis: se las damos para que las use con cada socio).
# pd.DataFrame enseña el resultado como una tabla.
pd.DataFrame(probar_casos(reglas, predecir, motor))

,caso,p,reglas usadas,decisión,esperada,¿bien?
0,Aitor,0.80,R2 -> R5,LLAMADA DE LA DIRECTORA,LLAMADA DE LA DIRECTORA,BIEN
1,Nerea,0.73,R2 -> R6,SESIÓN GRATIS,SESIÓN GRATIS,BIEN
2,Unai,0.55,R3 -> R7,EMAIL CON DESCUENTO,EMAIL CON DESCUENTO,BIEN
3,Irati,0.27,R4 -> R8,NO HACER NADA,NO HACER NADA,BIEN
4,Gorka,0.97,R1,REVISAR FICHA (dato erróneo),REVISAR FICHA (dato erróneo),BIEN
5,Lander,0.73,R2 -> R6,SESIÓN GRATIS,SESIÓN GRATIS,BIEN
6,Maite,0.80,R2 -> R9,LLAMADA + MES GRATIS,LLAMADA + MES GRATIS,BIEN
7,Ane,0.55,R10,PASAR A ADMINISTRACIÓN,PASAR A ADMINISTRACIÓN,BIEN
8,Mikel,0.80,R2 -> R11,AVISAR A LA FAMILIA,AVISAR A LA FAMILIA,BIEN
9,Jon,0.45,R12,REVISAR FICHA (dato erróneo),REVISAR FICHA (dato erróneo),BIEN


Deberían salir los 5 en BIEN. Fíjate en Gorka: el modelo le da una probabilidad muy alta, pero gana R1, la regla de seguridad, porque sus visitas son imposibles.

**Si algún día sale MAL**, hay dos posibilidades:
- la regla está mal escrita (por ejemplo, un `>=` donde debía ir un `<`), o
- lo que esperabas estaba mal pensado y el sistema tiene razón.

Mira la columna *reglas usadas* y sigue el caso vuelta a vuelta, como en el E2.

### Tarea 1. Añadir un caso límite

Los fallos suelen estar en los **límites**: justo en 0,7, ¿es riesgo alto o medio? Vamos a añadir un caso justo ahí.

1. Ejecuta la celda de abajo: te dice qué probabilidad da el modelo según las visitas.
2. Elige un número de visitas cuya probabilidad esté cerca de 0,7.
3. Abre `mia/casos.py` y añade una línea a la lista `CASOS` con ese socio y la decisión que esperas. Copia una de las que hay y cámbiala.
4. Guarda el fichero, **reinicia el kernel** y ejecuta desde el Paso 1 hasta aquí.

In [5]:
for visitas in range(0, 11):
    print(visitas, 'visitas al mes -> p =', predecir({'visitas_mes': visitas}))

0 visitas al mes -> p = 0.9
1 visitas al mes -> p = 0.86
2 visitas al mes -> p = 0.8
3 visitas al mes -> p = 0.73
4 visitas al mes -> p = 0.65
5 visitas al mes -> p = 0.55
6 visitas al mes -> p = 0.45
7 visitas al mes -> p = 0.36
8 visitas al mes -> p = 0.27
9 visitas al mes -> p = 0.2
10 visitas al mes -> p = 0.15


## Paso 4. Cambiar un límite sin tocar el modelo

Igual que con Leire en el E2. La directora decide que el riesgo alto empieza en 0,6 en vez de en 0,7.

En el E2 cambiábamos el número en una celda. Ahora el límite está en el fichero `mia/reglas.py`, así que se cambia **allí**. Es como lo haréis en vuestro proyecto.

In [6]:
from mia.reglas import LIMITE_ALTO     # para ver qué límite está usando ahora reglas.py

leire = {'nombre': 'Leire', 'visitas_mes': 4, 'meses_socio': 30}
leire['p'] = predecir(leire)

print('LIMITE_ALTO en reglas.py:', LIMITE_ALTO)
print('p de Leire:', leire['p'])
motor(leire, reglas, ver=True);

LIMITE_ALTO en reglas.py: 0.7
p de Leire: 0.65
Vuelta 1 : se cumplen ['R3'] -> se aplica R3 -> apunta {'riesgo': 'medio'}
Vuelta 2 : se cumplen ['R7'] -> se aplica R7 -> apunta {'decision': 'EMAIL CON DESCUENTO'}

Leire : EMAIL CON DESCUENTO   (reglas usadas: R3 -> R7 )


### Tarea 2.  La directora baja el límite a 0,6

1. Ejecuta la celda de arriba: Leire tiene p = 0,65 y, con el límite en 0,7, recibe un **email con descuento** (R3 → R7).
2. Abre `mia/reglas.py` y cambia `LIMITE_ALTO = 0.7` por `LIMITE_ALTO = 0.6`. Guarda el fichero.
3. **Reinicia el kernel** (Kernel → Restart) y ejecuta desde el Paso 1 hasta aquí.
4. Ahora Leire recibe una **llamada de la directora** (R2 → R5).

**Pregunta:** ¿ha cambiado la probabilidad de Leire? ¿Has tenido que tocar el modelo?

Cuando termines, vuelve a poner `LIMITE_ALTO = 0.7` en `mia/reglas.py`, guarda y reinicia el kernel.

La probabilidad de Leire **no ha cambiado** (sigue siendo 0,65): el modelo no se ha tocado. Lo que ha cambiado es el número a partir del cual el gimnasio considera que el riesgo es alto. Por eso pasa de un email a una llamada.

**La idea para el Reto 1:** el modelo lo entrenáis una vez. Las reglas las puede ajustar la empresa cuando quiera, sin reentrenar nada.

## Paso 5. Nuevas reglas para el gimnasio

### La situación

La directora de KirolGym quiere usar **más datos de cada socio**. Son datos que el gimnasio ya tiene en la ficha, pero que **el modelo no usa** (el modelo solo mira las visitas):

| Dato | Valores posibles | Ejemplo |
|---|---|---|
| `edad` | un número | `16` |
| `tipo_cuota` | `'basica'` o `'premium'` | `'premium'` |
| `cuota_impagada` | `'si'` o `'no'` | `'si'` |

**La idea importante:** el modelo predice el riesgo, pero la empresa sabe cosas que el modelo no sabe. Esas cosas se meten en las reglas, sin tocar el modelo.

### Las 4 reglas nuevas que pide la directora

| Regla | SI… | ENTONCES… | Prioridad | Por qué esa prioridad |
|---|---|---|---|---|
| R9 | riesgo alto **Y** es premium | LLAMADA + MES GRATIS | 2 | Tiene que ganar a R5 y R6 (prioridad 1), que también se cumplen con riesgo alto |
| R10 | debe alguna cuota | PASAR A ADMINISTRACIÓN | 8 | Si debe dinero, no se le ofrece nada, tenga el riesgo que tenga |
| R11 | es menor de edad **Y** riesgo alto | AVISAR A LA FAMILIA | 5 | Con un menor, primero se habla con la familia, antes que cualquier oferta |
| R12 | la edad es imposible (menor que 0 o mayor que 100) | REVISAR FICHA (dato erróneo) | 10 | Es una regla de seguridad, como R1: con un dato imposible no se decide nada |

Las vais a escribir en dos ficheros, siguiendo los pasos de abajo **en orden**.

### Tarea 3. Escribir las condiciones en `mia/reglas.py`

**3.1 · Abrid el fichero.** En el explorador de Jupyter (o de VS Code), entrad en la carpeta `mia` y abrid `reglas.py`.

**3.2 · Buscad la función `si_R8`.** Pulsad `Ctrl + F` y escribid `def si_R8`. Es la última función, justo antes de la lista de reglas.

**3.3 · Debajo de `si_R8`, pegad la función de R9** (ya está hecha, solo copiadla):

```python
def si_R9(d):
    """R9. ¿Riesgo alto Y el socio es premium?"""
    return d.get('riesgo') == 'alto' and d.get('tipo_cuota') == 'premium'
```

**3.4 · Escribid vosotros R10, R11 y R12** debajo de R9. Copiad R9 y cambiad la línea del `return`:

```python
def si_R10(d):
    """R10. ¿El socio debe alguna cuota?  cuota_impagada"""
    return ...


def si_R11(d):
    """R11. ¿Es menor de edad Y tiene riesgo alto?"""
    return ...


def si_R12(d):
    """R12 (SEGURIDAD). ¿La edad es imposible?"""
    return ...
```

**Pistas:**

| Regla | Pista |
|---|---|
| R10 | Los textos se comparan entre comillas: `d.get('cuota_impagada') == 'si'` |
| R11 | "Menor de edad" es `d.get('edad', 30) < 18`. El `30` es la edad que se usa si el socio no la tiene apuntada: así la regla no se cumple con Aitor, Nerea... Luego añadid `and` y el riesgo, como en R9. |
| R12 | Hay que usar `or`: la edad es menor que 0 **o** mayor que 100. Usad también `d.get('edad', 30)`. |

**¿Por qué siempre `d.get(...)`?** Porque los socios antiguos (Aitor, Nerea...) no tienen `edad`, ni `tipo_cuota`, ni `cuota_impagada`. Con `d.get`, si el dato no está, la regla simplemente no se cumple y no da error.

### Tarea 4. Añadir las reglas a la lista, en el mismo fichero

**4.1 · Bajad hasta la lista `reglas = [`**, al final de `reglas.py`.

**4.2 · Antes del corchete de cierre `]`**, añadid una línea por cada regla nueva. La de R9 ya está hecha:

```python
    ('R9',  si_R9,  {'decision': 'LLAMADA + MES GRATIS'},  2, 'riesgo alto y es socio premium'),
```

Las otras tres las completáis vosotros, cambiando los `...`:

```python
    ('R10', si_R10, {'decision': ...}, ..., '...'),
    ('R11', si_R11, {'decision': ...}, ..., '...'),
    ('R12', si_R12, {'decision': ...}, ..., '...'),
```

Recordad los cinco datos, siempre en este orden: **nombre, condición, qué apunta, prioridad, porqué**. La decisión y la prioridad están en la tabla de arriba. El porqué es una frase corta que explique la regla.

**4.3 · Guardad el fichero** (`Ctrl + S`).

### Tarea 5. Añadir un socio de prueba por cada regla en `mia/casos.py`

**5.1 · Abrid `mia/casos.py`** (también está en la carpeta `mia`).

**5.2 · Buscad la lista `CASOS = [`.** Antes del corchete de cierre `]`, añadid estos cuatro socios. Maite ya está completa. En los otros tres, **escribid vosotros la decisión que esperáis** en lugar de los `...`:

```python
    ('Maite', {'visitas_mes': 2, 'meses_socio': 10, 'tipo_cuota': 'premium'},  'LLAMADA + MES GRATIS'),
    ('Ane',   {'visitas_mes': 5, 'meses_socio': 40, 'cuota_impagada': 'si'},   ...),
    ('Mikel', {'visitas_mes': 2, 'meses_socio': 12, 'edad': 16},               ...),
    ('Jon',   {'visitas_mes': 6, 'meses_socio': 20, 'edad': -5},               ...),
```

**Pensadlo antes de escribir:** Ane tiene riesgo medio y debe una cuota. ¿Qué regla gana? Mikel tiene 16 años y riesgo alto. ¿Qué regla gana, R6 o R11? Mirad las prioridades.

**5.3 · Guardad el fichero** (`Ctrl + S`).

### Tarea 6. Probarlo todo

**6.1 · Reiniciad el kernel:** menú *Kernel → Restart*. Sin esto, el notebook sigue usando los ficheros antiguos.

**6.2 · Ejecutad la celda del Paso 1.** Ahora tiene que decir: `Hay 12 reglas en mia/reglas.py`.

**6.3 · Ejecutad la celda del Paso 3.** Tienen que salir **9 casos, todos en BIEN**.

**Si algo falla:**

| Lo que pasa | Qué revisar |
|---|---|
| Sigue diciendo "Hay 8 reglas" | No habéis guardado `reglas.py` o no habéis reiniciado el kernel |
| `SyntaxError` o `IndentationError` | Falta una coma al final de una línea de la lista, un paréntesis, o los 4 espacios delante del `return` |
| `NameError: name 'si_R10' is not defined` | La función está mal escrita o se llama distinto en la lista |
| Un caso sale **MAL** | Mirad la columna *reglas usadas*: ¿se ha aplicado la regla que esperabais? Revisad la condición y la prioridad |

### Tarea 7. Un caso difícil

Un socio que cumple **varias reglas nuevas a la vez**: es premium, tiene riesgo alto **y** debe una cuota.


1. Ejecutad la celda de abajo y comprobadlo. Con `ver=True` veréis cada vuelta.
2. Después, cambiad los datos (por ejemplo, poned `'edad': 15` o `'cuota_impagada': 'no'`) y volved a ejecutar. ¿Cambia la decisión?

In [7]:
# Socio con varios datos a la vez. Cambiadlos para probar otros casos.
socio = {'nombre': 'Caso difícil', 'visitas_mes': 2, 'meses_socio': 10,
         'tipo_cuota': 'premium', 'cuota_impagada': 'si', 'edad': 30}

socio['p'] = predecir(socio)        # el modelo solo mira las visitas
motor(socio, reglas, ver=True);     # las reglas miran todo lo demás

Vuelta 1 : se cumplen ['R2', 'R10'] -> se aplica R10 -> apunta {'decision': 'PASAR A ADMINISTRACIÓN'}

Caso difícil : PASAR A ADMINISTRACIÓN   (reglas usadas: R10 )


### Preguntas

1. Con Ane, en la vuelta 1 se cumplen R3 y R10. ¿Por qué gana R10?
2. Con Mikel, en la vuelta 2 se cumplen R6 y R11. ¿Cuál gana y por qué?
3. Con el caso difícil, ¿qué regla ha ganado? ¿Es lo que querría la directora?
4. ¿Por qué no ha hecho falta volver a entrenar el modelo para añadir estas reglas?

### Antes de terminar, comprobad

| | ¿Lo tenéis? |
|---|---|
| El Paso 1 dice "Hay 12 reglas" | |
| Cada regla nueva tiene su frase del porqué | |
| El Paso 3 da 9 casos, todos en BIEN | |
| Habéis probado el caso difícil y sabéis explicar por qué sale esa decisión | |

**Esto es exactamente lo que haréis en el Reto 1** con vuestro proyecto: cambiar el gimnasio por vuestra empresa, y los datos y las reglas por los vuestros. Es lo que se evalúa del RA5: 6 reglas o más, probadas con casos, con una regla de seguridad, con su porqué, y saber explicar una decisión vuelta a vuelta.

## Paso 6. Conectar vuestro modelo de SAA

Cuando tengáis vuestro modelo entrenado en SAA:

1. **Guardadlo** en vuestro notebook de SAA:
   ```python
   import joblib
   joblib.dump(modelo, 'modelo.joblib')
   ```
2. **Copiad** `modelo.joblib` a la carpeta de la plantilla, al lado de `modelo.py`.
3. En `modelo.py`, poned en **`COLUMNAS`** las columnas con las que lo entrenasteis, en el mismo orden. Por ejemplo: `COLUMNAS = ['visitas_mes', 'meses_socio', 'edad']`.
4. **Reiniciad el kernel** y ejecutad el Paso 2 con un caso vuestro.

5. A partir de ahí, predecir() usa tu  modelo en vez del del gimnasio. No hay que hacer nada más: al arrancar, modelo.py comprueba si existe modelo.joblib y, si existe, lo carga.

Mientras no haya `modelo.joblib`, la plantilla usa el modelo del gimnasio.

## Paso 7 (opcional). ¿Y si vuestro modelo es de regresión?

En SAA habéis visto la **regresión lineal**. Un modelo de regresión no da una probabilidad: da **un número** (euros, días, visitas…).

**Las reglas funcionan igual.** Lo único que cambia son las reglas que traducen el número a una palabra (aquí R2, R3 y R4). El motor no cambia.

Ejemplo: KirolGym quiere predecir **cuántas visitas hará cada socio el mes que viene**. Si van a ser pocas, hay riesgo. *(La columna `visitas_prox` es inventada para el ejemplo.)*

In [8]:
import numpy as np
from sklearn.linear_model import LinearRegression

df = pd.read_csv('datos/gimnasio.csv')

# Columna inventada para el ejemplo: visitas del mes que viene
rng = np.random.default_rng(0)
df['visitas_prox'] = (df['visitas_mes'] - 3 * df['baja'] + rng.integers(-1, 2, len(df))).clip(lower=0)

# Regresión lineal, como en SAA: aprende a predecir las visitas del mes que viene
regresion = LinearRegression()
regresion.fit(df[['visitas_mes', 'meses_socio']], df['visitas_prox'])


def predecir_visitas(caso):
    """Visitas previstas el mes que viene (un número, no una probabilidad)."""
    fila = pd.DataFrame([{'visitas_mes': caso['visitas_mes'], 'meses_socio': caso['meses_socio']}])
    return float(round(regresion.predict(fila)[0], 1))


print('Aitor (2 visitas) ->', predecir_visitas(aitor), 'visitas previstas')
print('Irati (8 visitas) ->', predecir_visitas({'visitas_mes': 8, 'meses_socio': 20}), 'visitas previstas')

Aitor (2 visitas) -> 0.0 visitas previstas
Irati (8 visitas) -> 6.8 visitas previstas


Ahora R2, R3 y R4 miran las **visitas previstas**. Ojo: al revés que con la probabilidad, **pocas visitas significa riesgo alto**.

Los signos van al revés que con la probabilidad. Ahora el número bajo es lo peligroso: viene poco al gimnasio.


Tres condiciones nuevas. Hacen lo mismo que R2, R3 y R4, pero miran el número de visitas previstas en vez de la probabilidad:

| Regla | Con clasificación (probabilidad) | Con regresión (visitas previstas) |
|---|---|---|
| R2 → riesgo alto | p ≥ 0,7 | menos de 3 visitas |
| R3 → riesgo medio | 0,4 ≤ p < 0,7 | entre 3 y 6 |
| R4 → riesgo bajo | p < 0,4 | 6 o más |

R1 y R5 a R8 **no cambian**: siguen leyendo el riesgo apuntado. El predicho lo guardamos en la ficha como `prevision`, en vez de `p`.

In [9]:
def si_R2_reg(d):
    return d['prevision'] < 3

def si_R3_reg(d):
    return d['prevision'] >= 3 and d['prevision'] < 6

def si_R4_reg(d):
    return d['prevision'] >= 6

# Copiamos las reglas y cambiamos solo R2, R3 y R4 (las posiciones 1, 2 y 3 de la lista)
reglas_reg = list(reglas)
reglas_reg[1] = ('R2', si_R2_reg, {'riesgo': 'alto'},  1, 'el modelo prevé menos de 3 visitas')
reglas_reg[2] = ('R3', si_R3_reg, {'riesgo': 'medio'}, 1, 'el modelo prevé entre 3 y 6 visitas')
reglas_reg[3] = ('R4', si_R4_reg, {'riesgo': 'bajo'},  1, 'el modelo prevé 6 visitas o más')

aitor_reg = {'nombre': 'Aitor', 'visitas_mes': 2, 'meses_socio': 36}
aitor_reg['prevision'] = predecir_visitas(aitor_reg)     # la regresión lineal predice
r = motor(aitor_reg, reglas_reg, ver=True)                # el MISMO motor decide
print(r['porque'])

Vuelta 1 : se cumplen ['R2'] -> se aplica R2 -> apunta {'riesgo': 'alto'}
Vuelta 2 : se cumplen ['R5'] -> se aplica R5 -> apunta {'decision': 'LLAMADA DE LA DIRECTORA'}

Aitor : LLAMADA DE LA DIRECTORA   (reglas usadas: R2 -> R5 )
['R2: el modelo prevé menos de 3 visitas', 'R5: riesgo alto y lleva 24 meses o más: es un socio fiel']


**Importante:** da igual qué modelo uséis (regresión lineal, regresión logística, árbol, random forest...). El modelo pone un número en la ficha y las reglas lo convierten en una decisión.

Si vuestro modelo es de regresión, en `modelo.py` hay que cambiar una línea. Está explicado dentro de la función `predecir()`.

## Lo que os lleváis

1. El Reto 1 es el E2 repartido en ficheros: **modelo** (`modelo.py`), **reglas** (`reglas.py`), **motor** (`motor.py`) y **casos** (`casos.py`).
2. Todo se resume en tres líneas: datos del caso → el modelo predice → las reglas deciden.
3. Vuestro trabajo de MIA son dos ficheros: `reglas.py` y `casos.py`.
4. Cada regla lleva su **porqué**, para que la app pueda explicar cada decisión.
5. Después de cambiar un `.py`, **reiniciad el kernel**.